# Federal AI Use Case Inventory - Exploration Notebook

Interactive analysis of the 2024 Federal Agency AI Use Case Inventory.

**Data Source:** OMB Memorandum M-24-10 Agency Submissions

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

In [ ]:
# Load the data
DATA_DIR = Path('/data')

# Load 2024 inventory
df_2024 = pd.read_csv(DATA_DIR / '2024_consolidated_ai_inventory_raw_v2.csv', low_memory=False)
print(f'2024 Inventory: {len(df_2024):,} use cases, {len(df_2024.columns)} columns')

# Load 2023 inventory for comparison
try:
    df_2023 = pd.read_csv(DATA_DIR / '2023_consolidated_ai_inventory_raw.csv', low_memory=False)
    print(f'2023 Inventory: {len(df_2023):,} use cases')
except FileNotFoundError:
    df_2023 = None
    print('2023 data not available')

In [ ]:
# Preview the data
df_2024.head()

In [ ]:
# Column names
print('Available columns:')
for i, col in enumerate(df_2024.columns, 1):
    print(f'{i:2}. {col}')

## Summary Statistics

In [ ]:
# Find key columns
agency_col = [c for c in df_2024.columns if 'Agency' in c and 'Abbreviation' not in c][0]
topic_col = [c for c in df_2024.columns if 'Topic' in c][0]
stage_col = [c for c in df_2024.columns if 'Stage' in c][0]
impact_col = [c for c in df_2024.columns if 'impacting' in c.lower()][0]

print(f'Agency column: {agency_col}')
print(f'Topic column: {topic_col}')
print(f'Stage column: {stage_col}')
print(f'Impact column: {impact_col}')

In [ ]:
# Summary stats
print('=== 2024 Federal AI Use Case Inventory Summary ===')
print(f'Total Use Cases: {len(df_2024):,}')
print(f'Unique Agencies: {df_2024[agency_col].nunique()}')
print(f'Topic Areas: {df_2024[topic_col].nunique()}')
print()

# Impact breakdown
print('Impact Classification:')
print(df_2024[impact_col].value_counts())

## Agency Analysis

In [ ]:
# Top agencies by use case count
agency_counts = df_2024[agency_col].value_counts().head(15)

fig = px.bar(
    x=agency_counts.values,
    y=agency_counts.index,
    orientation='h',
    title='Top 15 Agencies by AI Use Case Count',
    labels={'x': 'Number of Use Cases', 'y': 'Agency'}
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Agency distribution table
agency_summary = df_2024.groupby(agency_col).agg(
    total_use_cases=('Use Case Name', 'count'),
).sort_values('total_use_cases', ascending=False)

agency_summary

## Topic Area Analysis

In [ ]:
# Topic distribution
topic_counts = df_2024[topic_col].value_counts()

fig = px.pie(
    values=topic_counts.values,
    names=topic_counts.index,
    title='AI Use Cases by Topic Area',
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=600)
fig.show()

## Development Stage Analysis

In [ ]:
# Development stages
stage_counts = df_2024[stage_col].value_counts()

fig = px.bar(
    x=stage_counts.index,
    y=stage_counts.values,
    title='AI Use Cases by Development Stage',
    labels={'x': 'Stage', 'y': 'Count'},
    color=stage_counts.values,
    color_continuous_scale='Blues'
)
fig.update_layout(showlegend=False)
fig.show()

## Impact Analysis

In [ ]:
# Rights/Safety impacting analysis
impact_counts = df_2024[impact_col].value_counts()

colors = ['#10B981', '#F59E0B', '#EF4444', '#3B82F6']

fig = px.pie(
    values=impact_counts.values,
    names=impact_counts.index,
    title='AI Use Cases by Impact Classification',
    color_discrete_sequence=colors
)
fig.update_traces(textposition='inside', textinfo='percent+value')
fig.show()

In [ ]:
# High-impact use cases by agency
high_impact_mask = df_2024[impact_col].str.lower().str.contains('rights|safety|both', na=False)
high_impact_df = df_2024[high_impact_mask]

print(f'Total high-impact use cases: {len(high_impact_df):,}')
print()
print('Top 10 agencies with high-impact AI use cases:')
high_impact_df[agency_col].value_counts().head(10)

## Year-over-Year Comparison

In [ ]:
if df_2023 is not None:
    print('=== Year-over-Year Comparison ===')
    print(f'2023 Use Cases: {len(df_2023):,}')
    print(f'2024 Use Cases: {len(df_2024):,}')
    print(f'Growth: {len(df_2024) - len(df_2023):,} ({(len(df_2024) - len(df_2023)) / len(df_2023) * 100:.1f}%)')
    
    # Comparison chart
    fig = go.Figure(data=[
        go.Bar(name='2023', x=['Use Cases'], y=[len(df_2023)], marker_color='#6B7280'),
        go.Bar(name='2024', x=['Use Cases'], y=[len(df_2024)], marker_color='#3B82F6')
    ])
    fig.update_layout(title='AI Use Case Growth: 2023 vs 2024', barmode='group')
    fig.show()
else:
    print('2023 data not available for comparison')

## Custom Analysis

Use the cells below for your own exploration.

In [ ]:
# Example: Filter by agency
# agency_filter = 'Department of Health and Human Services'
# df_2024[df_2024[agency_col] == agency_filter].head()

In [ ]:
# Example: Search use cases by keyword
# keyword = 'machine learning'
# name_col = 'Use Case Name'
# df_2024[df_2024[name_col].str.lower().str.contains(keyword, na=False)]

In [ ]:
# Your analysis here
